In [4]:
!pip install ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 914.9/914.9 kB 3.3 MB/s  0:00:00m 3.3 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 4.7 MB/s  0:00:00m 4.6 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [ipywidgets]


In [6]:
# 1. Install the missing package silently
!pip install kagglehub -q

import os
import kagglehub
import json
import csv
import random


In [8]:

print("============== KAGGLE PIPELINE INITIATED ==============")

# 2. Authenticate with Kaggle (Replace with your actual credentials!)
# os.environ['KAGGLE_USERNAME'] = "YOUR_KAGGLE_USERNAME_HE
os.environ['KAGGLE_KEY'] = "KGAT_c4f5618f0cf364f843cd06ea355b16cd"

print("Downloading 4GB Yelp Dataset directly to remote computer...")
print("(This may take a few minutes depending on the remote internet speed...)")

# 3. Download the dataset
dataset_folder = kagglehub.dataset_download("yelp-dataset/yelp-dataset")
print(f"✅ Download Complete! Saved to: {dataset_folder}")

# 4. Seamlessly route the downloaded folder into the RAM-Safe script
# The kagglehub returns a folder, so we target the specific JSON file inside it
raw_json_path = os.path.join(dataset_folder, 'yelp_academic_dataset_review.json')
output_csv_path = 'yelp_3_6M_imbalanced.csv'

============== KAGGLE PIPELINE INITIATED ==============
(This may take a few minutes depending on the remote internet speed...)


100%|█████████████████████████████████████| 4.07G/4.07G [2:07:25<00:00, 572kB/s]

Extracting files...


✅ Download Complete! Saved to: /home/asif/.cache/kagglehub/datasets/yelp-dataset/yelp-dataset/versions/4


In [9]:
import json
import csv
import random

print("Starting RAM-Safe Extraction for Imbalanced Yelp Dataset...")

raw_json_path = 'yelp_academic_dataset_review.json' # Path to downloaded raw Yelp file
output_csv_path = 'yelp_3_6M_imbalanced.csv'

# Your 70/30 targets
TARGET_POS = 2520000
TARGET_NEG = 1080000

pos_collected = 0
neg_collected = 0

# We will store the extracted rows here before saving
extracted_data = []

# Read line-by-line to prevent RAM crashes
with open(raw_json_path, 'r', encoding='utf-8') as infile:
    for line in infile:
        # Stop reading if we hit our exact targets
        if pos_collected == TARGET_POS and neg_collected == TARGET_NEG:
            break
            
        review = json.loads(line)
        stars = review['stars']
        text = review['text'].replace('\n', ' ') # Clean line breaks
        
        # 1-2 stars = Negative (Label 0)
        if stars <= 2 and neg_collected < TARGET_NEG:
            extracted_data.append({'text': text, 'label': 0})
            neg_collected += 1
            
        # 4-5 stars = Positive (Label 1)
        elif stars >= 4 and pos_collected < TARGET_POS:
            extracted_data.append({'text': text, 'label': 1})
            pos_collected += 1

print(f"Extraction Complete! Positive: {pos_collected} | Negative: {neg_collected}")
print("Shuffling data to prevent chronological bias...")
random.shuffle(extracted_data)

print("Writing to CSV...")
with open(output_csv_path, 'w', newline='', encoding='utf-8') as outfile:
    writer = csv.DictWriter(outfile, fieldnames=['text', 'label'])
    writer.writeheader()
    writer.writerows(extracted_data)

print(f"✅ Success! Saved to {output_csv_path}")

Starting RAM-Safe Extraction for Imbalanced Yelp Dataset...
Extraction Complete! Positive: 2520000 | Negative: 1080000
Shuffling data to prevent chronological bias...
Writing to CSV...
✅ Success! Saved to yelp_3_6M_imbalanced.csv
